In [11]:
from langgraph.graph import StateGraph,START,END
from langchain_openai import ChatOpenAI,OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.tools import tool
from typing import TypedDict,Annotated
from langgraph.graph.message import add_messages
from langchain_core.messages import HumanMessage,BaseMessage
from langgraph.prebuilt import ToolNode,tools_condition
from langchain_text_splitters import RecursiveCharacterTextSplitter,TextSplitter

In [12]:
load_dotenv()
model = ChatOpenAI()

In [13]:
class SubState(TypedDict):
    input_text:str
    translated_text:str

In [14]:
def translate_text(state:SubState):
    prompt=f"""
    
    Translate the following text in hindi.
    keep it natural and clear.Do not add extra content.
    
    Text:
    {state['input_text']}
    """.strip()
    
    translate_text_response = model.invoke(prompt).content
    return {'translated_text':translate_text_response}

In [15]:
subgraph = StateGraph(SubState)

subgraph.add_node('translate_text',translate_text)
subgraph.add_edge(START,'translate_text')
subgraph.add_edge('translate_text',END)

subgraph_workflow = subgraph.compile()

In [16]:
class ParentState(TypedDict):
    question:str
    ans_eng:str
    ans_hin:str

In [17]:
def generate_answer(state:ParentState):
    prompt = f"""
    You are a helpful assistant. Anwser clearly.
    \n\nQuestion: {state['question']}
    """
    response = model.invoke(prompt).content
    return{
        'ans_eng':response
    }

In [18]:
def translate_answer(state:ParentState):
    result = subgraph_workflow.invoke({'input_text':state['ans_eng']})
    return {
        'ans_hin':result['translated_text']
    }

In [19]:
parent_graph = StateGraph(ParentState)

parent_graph.add_node('answer',generate_answer)
parent_graph.add_node('translate',translate_answer)

parent_graph.add_edge(START,'answer')
parent_graph.add_edge('answer','translate')
parent_graph.add_edge('translate',END)

final_workflow = parent_graph.compile()

In [20]:
result1 = final_workflow.invoke({'question':'What is quantam physics'})

In [21]:
result1

{'question': 'What is quantam physics',
 'ans_eng': 'Answer: Quantum physics, or quantum mechanics, is a branch of physics that studies the behavior of particles at the smallest scales, including atoms and subatomic particles. It explains phenomena such as wave-particle duality, quantum entanglement, and superposition.',
 'ans_hin': 'उत्तर: क्वांटम भौतिकी, या क्वांटम मैकेनिक्स, वह शाखा है जो तत्वों के व्यवहार का अध्ययन करती है जो सबसे छोटे स्तरों पर होते हैं, जैसे कि परमाणु और उप-परमाणु कण। यह तथ्यों को समझाती है जैसे कि तरंग-कण द्व्यधिकया, क्वांटम उलझन, और उभिकरण।'}